# 1. Climate data extraction


In [1]:
from pathlib import Path
import zipfile
import numpy as np
import pandas as pd
import rasterio
import xarray as xr
from pyproj import Transformer, Geod

BASE_DIR = Path.cwd()
INPUT_FILE = BASE_DIR / 'meta_data_raw.xlsx'
OUTPUT_FILE = BASE_DIR / 'raster_derived_data.xlsx'
CLIMATE_DIR = BASE_DIR / 'raster' / 'climate_data'
SOIL_DIR = BASE_DIR / 'raster' / 'soil_data'
EXPECTED_ROWS = 708
ID_COLUMNS = ['Serial No.', 'Longitude', 'Latitude']
COORD_COLS = ['Longitude', 'Latitude']

raw_df = pd.read_excel(INPUT_FILE, sheet_name=0)
if EXPECTED_ROWS is not None:
    assert len(raw_df) == EXPECTED_ROWS
identifiers = raw_df[ID_COLUMNS].copy().reset_index(drop=True)
coords = raw_df[COORD_COLS].apply(pd.to_numeric, errors='coerce')
valid = (np.isfinite(coords).all(axis=1)
         & coords.Longitude.between(-180, 180)
         & coords.Latitude.between(-90, 90))
unique_coords = coords.loc[valid].drop_duplicates().reset_index(drop=True)
assert not unique_coords.empty
points = unique_coords.to_numpy(dtype=float)


def sample_raster(path):
    values = np.full(len(points), np.nan)
    with rasterio.open(path) as src:
        if src.crs is None or src.count != 1:
            raise ValueError(f'Invalid raster CRS or band count: {path}')
        transformer = Transformer.from_crs('EPSG:4326', src.crs, always_xy=True)
        x, y = transformer.transform(points[:, 0], points[:, 1])
        x, y = np.asarray(x), np.asarray(y)
        positions = np.flatnonzero(np.isfinite(x) & np.isfinite(y))
        rows, cols = rasterio.transform.rowcol(src.transform, x[positions], y[positions])
        rows, cols = np.asarray(rows), np.asarray(cols)
        inside = (rows >= 0) & (rows < src.height) & (cols >= 0) & (cols < src.width)
        positions = positions[inside][np.lexsort((cols[inside], rows[inside]))]
        for i, value in zip(positions, src.sample(zip(x[positions], y[positions]), masked=True)):
            if not np.ma.getmaskarray(value)[0] and np.isfinite(value[0]):
                values[i] = float(value[0]) * src.scales[0] + src.offsets[0]
    return values


climate_names = {
    'tavg': 'MAT_raster',
    'prec': 'MAP_raster',
    'srad': 'SRAD_raster',
    'vapr': 'VAPR_raster',
}
climate_unique = unique_coords.copy()
for var, column in climate_names.items():
    archive_path = CLIMATE_DIR / f'wc2.1_30s_{var}.zip'
    with zipfile.ZipFile(archive_path) as archive:
        members = {Path(name).name: name for name in archive.namelist()}
    monthly = []
    for month in range(1, 13):
        member = members[f'wc2.1_30s_{var}_{month:02d}.tif']
        monthly.append(sample_raster(f'/vsizip/{archive_path.as_posix()}/{member}'))
    monthly = np.stack(monthly)
    climate_unique[column] = monthly.sum(axis=0) if var == 'prec' else monthly.mean(axis=0)
    print(f'{column}: extraction completed.', flush=True)


MAT_raster: extraction completed.
MAP_raster: extraction completed.
SRAD_raster: extraction completed.
VAPR_raster: extraction completed.


# 2. Soil data extraction


In [2]:
soil_names = {
    'OC': 'SOC_i_raster',
    'PHH2O': 'pH_raster',
    'TN': 'TN_raster',
    'CLAY': 'Clay_raster',
    'CEC': 'CEC_raster',
}
soil_factors = {'OC': 0.1, 'PHH2O': 0.1, 'TN': 0.1, 'CLAY': 1.0, 'CEC': 1.0}
weights = np.array([4.5, 4.6, 7.5, 3.4]) / 20
MAX_CEC_FILL_KM = 10.0
geod = Geod(ellps='WGS84')
soil_unique = unique_coords.copy()
for var, column in soil_names.items():
    values = np.full(len(points), np.nan)
    with xr.open_dataset(SOIL_DIR / f'{var}1.nc') as ds:
        np.testing.assert_allclose(ds.depth.values, [4.5, 9.1, 16.6, 28.9])
        lon, lat = ds.lon.values.astype(float), ds.lat.values.astype(float)
        for i, (x, y) in enumerate(points):
            if not (lon.min() <= x <= lon.max() and lat.min() <= y <= lat.max()):
                continue
            col, row = np.abs(lon - x).argmin(), np.abs(lat - y).argmin()
            layers = ds[var].isel(lon=col, lat=row).values.astype(float)
            if np.isfinite(layers).all():
                values[i] = layers @ weights * soil_factors[var]
            elif var == 'CEC':
                lat_delta = MAX_CEC_FILL_KM / 110.0
                cos_lat = np.cos(np.deg2rad(min(abs(y) + lat_delta, 89.999)))
                lon_delta = min(180.0, MAX_CEC_FILL_KM / (110.0 * cos_lat))
                distance_lon = np.abs((lon - x + 180) % 360 - 180)
                cols = np.flatnonzero(distance_lon <= lon_delta)
                rows = np.flatnonzero(np.abs(lat - y) <= lat_delta)
                candidates = ds[var].isel(lat=rows, lon=cols).transpose('depth', 'lat', 'lon').values
                rr, cc = np.where(np.isfinite(candidates).all(axis=0))
                if len(rr):
                    sx, sy = lon[cols[cc]], lat[rows[rr]]
                    _, _, distance = geod.inv(np.full(len(rr), x), np.full(len(rr), y), sx, sy)
                    j = int(np.argmin(distance))
                    if distance[j] <= MAX_CEC_FILL_KM * 1000:
                        values[i] = candidates[:, rr[j], cc[j]].astype(float) @ weights
    soil_unique[column] = values
    print(f'{column}: extraction completed.', flush=True)


SOC_i_raster: extraction completed.
pH_raster: extraction completed.
TN_raster: extraction completed.
Clay_raster: extraction completed.
CEC_raster: extraction completed.


# 3. Export raster-derived data


In [3]:
derived_columns = list(climate_names.values()) + list(soil_names.values())
lookup = climate_unique.merge(soil_unique, on=COORD_COLS, how='inner', validate='one_to_one')
row_keys = coords.copy()
row_keys['_source_row'] = np.arange(len(raw_df))
mapped = row_keys.merge(lookup, on=COORD_COLS, how='left', sort=False, validate='many_to_one')
mapped = mapped.sort_values('_source_row').reset_index(drop=True)
assert mapped['_source_row'].tolist() == list(range(len(raw_df))), 'Record order or count changed'
result_df = pd.concat([identifiers, mapped[derived_columns]], axis=1)
assert len(result_df) == len(raw_df)
assert result_df.columns.tolist() == ID_COLUMNS + derived_columns
pd.testing.assert_frame_equal(result_df[ID_COLUMNS], identifiers)
assert mapped.groupby(COORD_COLS)[derived_columns].nunique(dropna=False).le(1).all().all()
result_df.to_excel(OUTPUT_FILE, sheet_name='raster_derived_data', index=False)

saved_df = pd.read_excel(OUTPUT_FILE)
pd.testing.assert_frame_equal(saved_df, result_df, check_dtype=False, check_exact=False)
print(f'Saved: {OUTPUT_FILE}')
print('Verified all records: original order and duplicate records preserved.')
if saved_df[derived_columns].notna().all().all():
    print('Raster values are available for all records.')
else:
    print('Missing raster values by output column:')
    print(saved_df[derived_columns].isna().sum().to_string())


Saved: c:\Meta\data_SD\meta_test\raster_derived_data.xlsx
Verified all records: original order and duplicate records preserved.
Raster values are available for all records.
